## Metode inference ini menggunakan pendekatan SHAP explainer sebagai fungsi utama reasoning.
## Codingan menggunakan metode dari "shap_explainer.joblib" daripada menggunakan "explanation_baselines.json".

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import shap
from pathlib import Path
from IPython.display import display

# Konfigurasi + Load Artifacts
def print_section(title):
    print("=" * 70)
    print(title)
    print("=" * 70)

# Tentukan path folder artifacts dan test data
artifact_dir = Path("../artifacts/post_award_anomaly")
test_data_dir = Path("../data/cleaned/test")

print_section("LOADING MASTER BUNDLE ARTIFACTS & TEST DATA")

# Load Master Model (Berisi model untuk semua region)
master_bundle = joblib.load(artifact_dir / "master_model.joblib")

print(f"Bundle Type      : {master_bundle['bundle_type']}")
print(f"Routing Column   : {master_bundle['routing_column']}")
print(f"Total Regions    : {master_bundle['region_count']} regions loaded.")

# Baca semua file CSV di dalam folder test data
test_files = list(test_data_dir.glob("*.csv"))
if not test_files:
    raise FileNotFoundError(f"Tidak menemukan file CSV di {test_data_dir}")

# Gabungkan semua data test dari berbagai daerah
all_test_data = pd.concat([pd.read_csv(f) for f in test_files], ignore_index=True)

# Ambil 10 sampel secara acak (random pick)
demo_input = all_test_data.sample(n=10, random_state=42).reset_index(drop=True)

# Tambahkan 'demo_case_label' dinamis agar log print di akhir tidak error
demo_input["demo_case_label"] = [f"Random_Test_Case_{i+1}" for i in range(len(demo_input))]

print(f"\nMemuat {len(demo_input)} kasus acak dari folder test untuk inference.")

In [ ]:
# Helper Functions
def format_number(value) -> str:
    """Human-readable number formatting used in explanation strings."""
    if pd.isna(value): return "missing"
    if isinstance(value, (int, np.integer)):     return f"{int(value):,}"
    if isinstance(value, (float, np.floating)):  return f"{value:,.4f}".rstrip("0").rstrip(".")
    return str(value)

def assign_severity(scores, medium_cutoff: float, anomaly_threshold: float):
    """Map raw anomaly scores to three severity bands: low / medium / high."""
    return np.select(
        [scores >= anomaly_threshold, scores >= medium_cutoff],
        ["high", "medium"],
        default="low",
    )

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add derived columns required by the model."""
    data = df.copy()
    data["date"]       = pd.to_datetime(data["date"],       errors="coerce")
    data["award_date"] = pd.to_datetime(data["award_date"], errors="coerce")

    # Calendar features from the award date
    data["award_month"]   = data["award_date"].dt.month
    data["award_quarter"] = data["award_date"].dt.quarter
    data["award_weekday"] = data["award_date"].dt.weekday

    # Log-transform to reduce skew on monetary values
    data["log_tender_minvalue"] = np.log1p(data["tender_minvalue"].clip(lower=0))
    data["log_award_value"]     = np.log1p(data["award_value"].clip(lower=0))

    # Derived monetary signal
    data["value_gap"] = data["award_value"] - data["tender_minvalue"]

    # Text-length features
    data["title_word_count"]       = data["tender_title"].fillna("").astype(str).str.split().str.len()
    data["award_title_word_count"] = data["award_title"].fillna("").astype(str).str.split().str.len()

    # Supplier participation
    tokens = data["award_supplier"].fillna("").astype(str).str.split(",")
    data["supplier_count"] = tokens.apply(
        lambda items: max(len([i for i in items if str(i).strip()]), 1)
    )

    # Speed of spend — guard against zero days_to_award
    safe_days = data["days_to_award"].replace(0, 1)
    data["award_value_per_day"] = data["award_value"] / safe_days
    data["same_day_award_flag"] = (data["days_to_award"] == 0).astype(int)

    return data

In [ ]:
# SHAP Explainer
# 1. Mapping Nama Fitur ke Bahasa Manusia (Business/Audit Terms)
KAMUS_KONSEP = {
    "award_title_word_count": "Kompleksitas Judul Kontrak",
    "days_to_award": "Durasi Proses Tender",
    "budget_utilization_ratio": "Rasio Penyerapan Anggaran",
    "value_gap": "Selisih Nilai Tender dan Kontrak",
    "supplier_count": "Jumlah Peserta Tender",
    "award_value_per_day": "Laju Pengeluaran Harian",
    "tender_minvalue": "Batas Minimum Tender",
    "award_value": "Nilai Kontrak Akhir",
    "mainprocurementcategory": "Kategori Pengadaan Utama"
}

# 2. Mapping Risiko Berdasarkan Analisis Audit
KAMUS_RISIKO = {
    "award_title_word_count": "pola penamaan judul yang tidak standar seringkali digunakan untuk mengaburkan spesifikasi asli proyek.",
    "days_to_award": "proses yang selesai terlalu cepat atau terlalu lambat mengindikasikan adanya potensi pengaturan pemenang (bid-rigging).",
    "budget_utilization_ratio": "penyerapan yang mendekati 100% secara sempurna merupakan indikator umum terjadinya mark-up harga.",
    "value_gap": "selisih yang tidak proporsional menunjukkan potensi inefisiensi atau kesalahan estimasi biaya awal.",
    "supplier_count": "minimnya partisipan tender dapat mengurangi kompetisi sehat dan meningkatkan risiko monopoli.",
    "award_value_per_day": "beban biaya harian yang ekstrem menunjukkan ketidakwajaran antara nilai proyek dengan durasi pengerjaan.",
    "tender_minvalue": "penetapan batas minimum yang tidak lazim berisiko membatasi partisipasi vendor kompeten.",
    "mainprocurementcategory": "anomali seringkali terpusat pada kategori spesifik ini akibat celah pada kebijakan atau kontrol internal yang kurang pengawasan."
}

def generate_natural_reason(feat, raw_val, shap_val, severity_band):
    """Menerjemahkan kontribusi SHAP menjadi narasi chatbot berdasarkan 3 tingkat risiko."""
    is_anomaly_driver = shap_val > 0
    nama_manusiawi = KAMUS_KONSEP.get(feat, feat.replace("_", " ").title())
    val_str = format_number(raw_val)
    
    # KASUS 1: RISIKO TINGGI (ANOMALY)
    if severity_band == "high":
        if is_anomaly_driver:
            risiko = KAMUS_RISIKO.get(feat, "hal ini memerlukan verifikasi kepatuhan dokumen lebih lanjut.")
            return (f"Sistem menemukan indikasi penyimpangan serius pada **{nama_manusiawi}** ({val_str}). "
                    f"Secara audit, {risiko}")
        else:
            return f"Meskipun terdapat temuan risiko lain, parameter **{nama_manusiawi}** ({val_str}) terpantau tetap stabil."

    # KASUS 2: RISIKO SEDANG (MEDIUM / BORDERLINE)
    elif severity_band == "medium":
        if is_anomaly_driver:
            return (f"Terdapat sedikit kejanggalan pada data **{nama_manusiawi}** ({val_str}). "
                    f"Meskipun menunjukkan pola yang tidak biasa, angka ini dinilai masih berada dalam batas toleransi kebijakan.")
        else:
            return f"Parameter **{nama_manusiawi}** ({val_str}) memberikan sinyal kestabilan di tengah beberapa anomali minor lainnya."

    # KASUS 3: RISIKO RENDAH (LOW / NORMAL)
    else:
        if is_anomaly_driver:
            return f"Komponen **{nama_manusiawi}** ({val_str}) menunjukkan aktivitas yang dinamis namun tetap sesuai dengan standar operasional."
        else:
            return f"Parameter **{nama_manusiawi}** ({val_str}) sangat identik dengan profil pengadaan yang bersih dan akuntabel."

def explain_prediction_shap(row_idx, row_shap, original_row, feature_names):
    """Mengambil 3 alasan utama berdasarkan SHAP values dengan membedakan 3 tingkatan bahasa."""
    severity_band = original_row['severity_band']
    
    # Ambil index dengan nilai SHAP absolut tertinggi (pendorong keputusan paling kuat)
    top_indices = np.argsort(np.abs(row_shap))[-3:][::-1]
    
    reasons = []
    for idx in top_indices:
        feat_name = feature_names[idx]
        # Mapping ulang ke kolom asli jika fiturnya hasil one-hot encoding
        raw_feat_name = "mainprocurementcategory" if feat_name.startswith("cat_") else feat_name
        raw_val = original_row[raw_feat_name] if raw_feat_name in original_row else "N/A"
        
        reasons.append(generate_natural_reason(raw_feat_name, raw_val, row_shap[idx], severity_band))
    
    # Header narasi berdasarkan tingkat risiko
    if severity_band == "high":
        header = "Sistem mendeteksi aktivitas yang MENCURIGAKAN dan berisiko tinggi:"
    elif severity_band == "medium":
        header = "Sistem menemukan beberapa temuan BORDERLINE yang memerlukan perhatian moderat:"
    else:
        header = "Status transaksi dinilai AMAN dan memenuhi kriteria kepatuhan standar:"

    summary = f"[{severity_band.upper()}] {header}\n" + "\n".join([f"• {r}" for r in reasons])
    return summary

In [ ]:
print_section("RUNNING ROUTED INFERENCE")

# 1. Engineering Fitur Global
demo_features = engineer_features(demo_input)

routing_col = master_bundle["routing_column"]
results = []

# 2. Loop berdasarkan Daerah (Grouping and Routing)
for region_name, group in demo_features.groupby(routing_col):
    routing_key = str(region_name).strip().casefold()
    
    if routing_key not in master_bundle["regions"]:
        print(f"Warning: Model tidak ditemukan untuk daerah '{region_name}'. Data ini dilewati.")
        continue
        
    # Extract region-specific objects
    region_bundle = master_bundle["regions"][routing_key]
    model = region_bundle["model"]
    preprocessor = region_bundle["preprocessor"]
    explainer_shap = region_bundle["explainer"]
    config = region_bundle["model_config"]
    meta = region_bundle["explanation_meta"]
    
    feature_columns = config["feature_columns"]
    feature_names_preprocessed = meta["feature_names_preprocessed"]
    
    # Transform & Prediksi Spesifik per Daerah
    X_group = preprocessor.transform(group[feature_columns])
    group_scores = -model.score_samples(X_group)
    
    # ---> KALKULASI TINGKAT ANOMALITAS / KEAMANAN <---
    baseline_score = config["medium_cutoff"]
    max_risk_score = config["anomaly_threshold"]
    
    # Skala perhitungan dibiarkan natural tanpa batasan (clip)
    # Jika hasil minus -> Berarti berada di zona AMAN (di bawah medium_cutoff)
    raw_risk_pct = ((group_scores - baseline_score) / (max_risk_score - baseline_score)) * 100
    
    # SHAP Explainer
    shap_values = explainer_shap.shap_values(X_group)
    if isinstance(shap_values, list):
        shap_values = shap_values[0]
        
    group_out = group.copy().reset_index(drop=True)
    group_out["anomaly_score"] = group_scores
    group_out["anomaly_risk_pct"] = np.round(raw_risk_pct, 2)
    
    # Format string agar lebih komunikatif
    def format_risk_string(val):
        if val < 0:
            return f"{abs(val)}%"
        elif val >= 100:
            return f"+{abs(val)}%"
        else:
            return f"+{abs(val)}%"
            
    group_out["anomaly_risk_str"] = group_out["anomaly_risk_pct"].apply(format_risk_string)
    
    group_out["prediction_label"] = np.select(
        [group_scores >= config["anomaly_threshold"], group_scores >= config["medium_cutoff"]],
        ["anomaly", "warning"],
        default="normal"
    )
    group_out["severity_band"] = assign_severity(group_scores, config["medium_cutoff"], config["anomaly_threshold"])
    
    # Generate Penjelasan Chatbot
    explanations = []
    for i in range(len(group_out)):
        reason_text = explain_prediction_shap(i, shap_values[i], group_out.iloc[i], feature_names_preprocessed)
        explanations.append(reason_text)
        
    group_out["human_readable_explanation"] = explanations
    results.append(group_out)

# 3. Gabungkan Semua Hasil
final_output = pd.concat(results, ignore_index=True)

# Jadikan 'nama_daerah' (provinsi) sebagai index agar print lebih rapi
final_output.set_index('nama_daerah', inplace=True)

# Tampilkan Hasil Akhir dengan kolom anomaly_risk_str yang sudah diformat
display_cols = [
    "demo_case_label", "award_date", "mainprocurementcategory", 
    "award_value", "anomaly_score", "anomaly_risk_str", "prediction_label", "human_readable_explanation"
]
display(final_output[display_cols])

# Simpan hasil
output_path = artifact_dir / "master_inference_output_final.csv"
final_output.to_csv(output_path, index=True)
print(f"\nHasil inference disimpan ke: {output_path}")

In [ ]:
print_section("DETAILED CHATBOT EXPLANATIONS")

# 'idx' sekarang mewakili index nama daerah yang telah di set sebelumnya
for idx, row in final_output.iterrows():
    print(f"Kasus: {row['demo_case_label']} | Daerah: {idx}")
    print(f"Status: {row['prediction_label'].upper()} (Score: {row['anomaly_score']:.4f})")
    print(f"Tingkat Risiko: {row['severity_band'].upper()}")
    print(f"Analisis Sistem:\n{row['human_readable_explanation']}")
    print("-" * 70)